# LifeMate — full app menu, on-demand complete schemas

**Keep your current adapter. No Qwen or ASR training.** This Git-backed notebook compares your previous agent setting with two menu-based agents, then runs public inference with the validation winner.

The supplied 62-point notebook uses the same Qwen3.5-4B base family but explicitly loads **AIO `llm-lora-p1`**, non-thinking E1P, and Gipformer v4 + ITN. It is not an agent-only comparison against our imported topical Thinking adapter. As requested, this notebook keeps your existing adapter and its `apps` / `open_apps` / tool-call / `done` JSON protocol. It does not download AIO weights.

Your senior's recommendation is implemented as **an always-visible compact menu of every installed app**, including Vietnamese names, descriptions and tool names. Full argument schemas and return descriptions are loaded only for requested apps. Initial routing never removes an app from the menu. If schemas must leave the prompt to make room, the app can be opened again.


## 1. Existing files and fixed comparison inputs

Keep the same files already used for your 55.4 run:

```
MyDrive/Zalo-AI-Challenge-2026/
  imports/teammate_topic_20261109/best_adapter.zip
  imports/teammate_topic_20261109/reports.zip
  ASR-cp/best.pt
```

No new model download from your teammate is required. The base model still downloads automatically if not cached.

By default, public inference reuses the **exact saved ASR JSONL from the previous experiment**, verified against its fingerprint. This keeps both adapter and recognized text fixed. You may explicitly provide another complete transcript file or choose `gipformer`, but then a public-score change cannot be attributed only to the agent. ITN is not newly enabled by this notebook.

Commit and push the new notebook and modified/new `training_pipeline/` files before opening it in Colab. Choose **L4 or A100**. Use a fresh experiment name when changing code, settings, or inputs. Old notebooks remain available; do not update an active training run's pinned Git commit.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/Zalo-AI-Challenge-2026')
IMPORT_NAME = 'teammate_topic_20261109'  # Existing selected adapter export.
PREVIOUS_EXPERIMENT = 'topic_agent_compare_20261109_v2'  # The run that produced 55.4; edit if different.
REFERENCE_MODE = 'auto'  # Read previous winner, or explicitly 'baseline', 'enhanced', 'read_guard'.
EXPERIMENT_NAME = 'menu_agent_20261110_v1'
ADAPTER_ZIP = DRIVE_ROOT / 'imports' / IMPORT_NAME / 'best_adapter.zip'
REPORTS_ZIP = DRIVE_ROOT / 'imports' / IMPORT_NAME / 'reports.zip'
WORK_ROOT = '/content/lifemate_menu_work'


## 2. Clone the implementation

The notebook contains settings and workflow calls only. For reproducibility, replace `main` with your pushed Git commit SHA. An existing checkout is not silently pulled mid-experiment.


In [ ]:
import sys, subprocess, importlib
REPO_URL = 'https://github.com/votaquangnhat/LifeMate-Baseline.git'
REPO_REF = 'main'  # Prefer your pushed commit SHA.
PROJECT = Path('/content/lifemate-menu-agent')
if not PROJECT.exists():
    subprocess.run(['git','clone',REPO_URL,str(PROJECT)],check=True)
    subprocess.run(['git','-C',str(PROJECT),'checkout',REPO_REF],check=True)
revision = subprocess.check_output(['git','-C',str(PROJECT),'rev-parse','HEAD'],text=True).strip()
expected = subprocess.check_output(['git','-C',str(PROJECT),'rev-parse',REPO_REF],text=True).strip()
assert revision == expected, 'Checkout differs from REPO_REF; choose the intended commit explicitly.'
CODE_DIR = PROJECT / 'training_pipeline'
assert (CODE_DIR / 'topic_agent/menu_agent.py').is_file(), 'Push the new menu agent files before Colab.'
sys.path.insert(0,str(CODE_DIR))
for name in ('topic_workflow','topic_environment','topic_artifacts','colab_workflow'):
    sys.modules.pop(name,None)
importlib.invalidate_caches()
import topic_workflow
print('Code commit:',revision)


## 3. Import existing weights and prepare the runtime

Displays GPU name/memory, checks the existing adapter/config/tokenizer export and pins its package revisions. GPU stages run in separate subprocesses. This does not train or change the selected weights.

`REFERENCE_MODE='auto'` reads the previous comparison winner and verifies its adapter hash. If that comparison file is unavailable, explicitly set the actual mode used for your 55.4 submission; do not guess a different reference setting.


In [ ]:
SESSION = topic_workflow.prepare(
    drive_root=DRIVE_ROOT, import_name=IMPORT_NAME, experiment_name=EXPERIMENT_NAME,
    adapter_zip=ADAPTER_ZIP, reports_zip=REPORTS_ZIP, work_root=WORK_ROOT,
)
REFERENCE = topic_workflow.previous_winner(SESSION, PREVIOUS_EXPERIMENT) if REFERENCE_MODE == 'auto' else REFERENCE_MODE
assert REFERENCE in ('baseline','enhanced','read_guard'), 'Choose the actual previous reference setting.'
print('Same-adapter reference:', REFERENCE)
subprocess.run([SESSION['python'],'-m','unittest','discover',
                '-s',str(CODE_DIR/'topic_agent'),'-p','selftest_*.py'],check=True)


## 4. Compare the current agent with the new agents

| Variant | Behavior |
|---|---|
| Previous reference | Same setting selected by your earlier comparison; rerun under this environment. |
| `menu` | Complete installed-app menu, Vietnamese app identities, lazy full schemas including returns, reversible schema eviction, and regeneration before executing a call to an unloaded app. |
| `menu_recovery` | Same menu agent plus deterministic dates from the sample clock, bounded empty-result guidance, repeated-read protection, and at most one completion review when apps remain unopened. Writes are never blanket-deduplicated. |

Both menu variants use an 8192-token context, a 1536-token history budget and a 7168-token executor-prompt ceiling. Menu entries and complete loaded schemas are not silently clipped. If protected context cannot fit, the sample stops with a diagnostic. The original adapter's thinking/action budgets remain unchanged.

The completion review does **not** demand another action: it may return `done` again for a completed or unsupported request. Empty-search guidance does not rewrite names, numbers or dates, and does not impose a blanket limit across unrelated searches.

Validation uses the same original IDs and reference states; labels are read only for scoring and diagnostics after execution. Higher local score wins, ties retain the first/reference variant. This is a small, previously used validation set, not proof of public-score improvement. No public labels are used.

Optional: supply ASR transcripts for **exactly these validation IDs**, including all history-user audio. Leave the path empty for gold-text validation. If the ASR trained on these recordings, treat the resulting score as a diagnostic rather than an untouched held-out result.


In [ ]:
VALIDATION_ASR_JSONL = ''
COMPARISON = topic_workflow.compare_validation(SESSION, validation_transcripts=VALIDATION_ASR_JSONL,
                                               variants=(REFERENCE, 'menu', 'menu_recovery'))
print('Selected agent:', COMPARISON['chosen'])


## 5. Inspect actual app coverage

The quoted **76/313** is your senior's observation, not a measured result for this adapter. An app absent from the *initial* router choice is not automatically a failure: lazy agents are expected to open apps later.

`validation_app_coverage.json` distinguishes:

- required app schemas never opened during rollout;
- required apps opened but never used for a real call;
- full app coverage with remaining errors in tools, arguments, ordering or completion.

For the new menu agents, coverage records apps whose complete schemas were placed in a generation prompt, not merely scheduled to be loaded. For older variants, opening may still be followed by their legacy schema-trimming policy. A complete menu makes an app accessible; it does not guarantee the model will choose it.


In [ ]:
import json
for variant in (REFERENCE, 'menu', 'menu_recovery'):
    path = Path(SESSION['experiment']) / variant / 'validation_app_coverage.json'
    report = json.loads(path.read_text())
    print(variant, {k:v for k,v in report.items() if k != 'cases'})


## 6. Independent public inference

After Step 4 finishes, the next cell can run alone in a fresh Colab runtime. It restores the selected model and winning agent; it does not train anything.

Default ASR behavior: find the transcript file used by the previous experiment, verify its stored hash, and reuse it. If the previous experiment folder/name is different, edit `PREVIOUS_EXPERIMENT`. If its metadata is unavailable, enter the exact file in `TRANSCRIPTS_PATH`. To decode afresh, explicitly select `ASR_MODE='gipformer'`; it uses the same `ASR-cp/best.pt` path as before.

Completed samples are reused on interruption. Early stops are printed and complete, schema-valid submissions still export. Incomplete outputs or GPU OOMs must be resolved before publication; no calls are fabricated to fill missing rows.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/Zalo-AI-Challenge-2026')
IMPORT_NAME = 'teammate_topic_20261109'  # Existing selected adapter export.
PREVIOUS_EXPERIMENT = 'topic_agent_compare_20261109_v2'  # The run that produced 55.4; edit if different.
REFERENCE_MODE = 'auto'  # Read previous winner, or explicitly 'baseline', 'enhanced', 'read_guard'.
EXPERIMENT_NAME = 'menu_agent_20261110_v1'
ADAPTER_ZIP = DRIVE_ROOT / 'imports' / IMPORT_NAME / 'best_adapter.zip'
REPORTS_ZIP = DRIVE_ROOT / 'imports' / IMPORT_NAME / 'reports.zip'
WORK_ROOT = '/content/lifemate_menu_work'
MODE = 'auto'  # Saved validation winner; or explicitly reference/menu/menu_recovery.
ASR_MODE = 'transcripts'  # Reuse the 55.4 run's recognition for an agent-only comparison.
TRANSCRIPTS_PATH = ''  # Empty: resolve verified previous ASR file. Otherwise an explicit JSONL path.
ASR_CHECKPOINT = str(DRIVE_ROOT / 'ASR-cp/best.pt')
ASR_DECODER = 'auto'

import sys, subprocess, importlib
REPO_URL = 'https://github.com/votaquangnhat/LifeMate-Baseline.git'
REPO_REF = 'main'  # Prefer your pushed commit SHA.
PROJECT = Path('/content/lifemate-menu-agent')
if not PROJECT.exists():
    subprocess.run(['git','clone',REPO_URL,str(PROJECT)],check=True)
    subprocess.run(['git','-C',str(PROJECT),'checkout',REPO_REF],check=True)
revision = subprocess.check_output(['git','-C',str(PROJECT),'rev-parse','HEAD'],text=True).strip()
expected = subprocess.check_output(['git','-C',str(PROJECT),'rev-parse',REPO_REF],text=True).strip()
assert revision == expected, 'Checkout differs from REPO_REF; choose the intended commit explicitly.'
CODE_DIR = PROJECT / 'training_pipeline'
assert (CODE_DIR / 'topic_agent/menu_agent.py').is_file(), 'Push the new menu agent files before Colab.'
sys.path.insert(0,str(CODE_DIR))
for name in ('topic_workflow','topic_environment','topic_artifacts','colab_workflow'):
    sys.modules.pop(name,None)
importlib.invalidate_caches()
import topic_workflow
print('Code commit:',revision)

SESSION = topic_workflow.prepare(
    drive_root=DRIVE_ROOT, import_name=IMPORT_NAME, experiment_name=EXPERIMENT_NAME,
    adapter_zip=ADAPTER_ZIP, reports_zip=REPORTS_ZIP, work_root=WORK_ROOT,
)

if ASR_MODE == 'transcripts' and not TRANSCRIPTS_PATH:
    prior_mode = topic_workflow.previous_winner(SESSION, PREVIOUS_EXPERIMENT) if REFERENCE_MODE == 'auto' else REFERENCE_MODE
    TRANSCRIPTS_PATH = topic_workflow.previous_transcripts(SESSION, PREVIOUS_EXPERIMENT, prior_mode)
print('Public transcript source:', TRANSCRIPTS_PATH if ASR_MODE == 'transcripts' else ASR_CHECKPOINT)
SUBMISSION = topic_workflow.public(SESSION, mode=MODE, asr_mode=ASR_MODE,
    transcripts_path=TRANSCRIPTS_PATH, asr_checkpoint=ASR_CHECKPOINT, asr_decoder=ASR_DECODER)
print('Ready to submit:', SUBMISSION)


## 7. Saved results

Under `MyDrive/Zalo-AI-Challenge-2026/runs/menu_agent_20261110_v1/`:

- `agent_comparison.json`: same-adapter validation scores and winner;
- each variant's `validation_app_coverage.json`, `validation_error_analysis.json`, and `validation_error_cases.jsonl`;
- the winner's `public_test_best/submission.jsonl` and `predictions_with_trace.jsonl`;
- public quality diagnostics, schema evictions, completion-review counts and early-stop reasons.

The latest validated file is also copied to `MyDrive/Zalo-AI-Challenge-2026/submission.jsonl`. Original model/import files and earlier experiment outputs are preserved.

Local tests exercise missing-app recovery, complete schema eviction/reloading, bounded completion review, preservation of repeated writes, and sample-clock date handling. GPU memory, runtime and public-score gains still require your Colab run. Read `notebooks/menu_agent_review.md` for the source comparison and limitations.


In [ ]:
from google.colab import files
files.download(str(SUBMISSION))
